# Блокнот 6. Обучение без учителя

Курс «Машинное обучение с нуля», модуль 6. Разделы: 1 — урок 6.1, 2–4 — урок 6.2, 5 — урок 6.3, 6 — урок 6.4, 7 — урок 6.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, HDBSCAN, KMeans
from sklearn.datasets import load_digits, load_sample_image, make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(0)

## 1. K-means своими руками (урок 6.1)

**Задание 1.** Допишите один шаг алгоритма Ллойда: отнести каждый объект к ближайшему центру и перенести центры в средние своих объектов (пустой кластер — центр остаётся на месте).

In [ ]:
def kmeans_step(X, centers):
    return ..., ...  # ваш код: новые центры и номера кластеров

In [ ]:
X, _ = make_blobs(300, centers=4, cluster_std=0.9, random_state=7)
centers = X[rng.choice(len(X), 4, replace=False)]
for step in range(100):
    new_centers, labels = kmeans_step(X, centers)
    if np.allclose(new_centers, centers):
        break
    centers = new_centers
inertia = ((X - centers[labels]) ** 2).sum()
sk = KMeans(4, random_state=0).fit(X)
print(f"Свой K-means: {step} шагов, инерция {inertia:.1f}; scikit-learn: {sk.inertia_:.1f}")
assert inertia >= sk.inertia_ - 1e-6, "инерция не может быть меньше лучшего решения"
plt.scatter(*X.T, c=labels, s=10, cmap="tab10")
plt.scatter(*centers.T, marker="X", s=200, c="black")
plt.title("Свой K-means")
plt.show()

Если ваша инерция заметно больше, чем у scikit-learn, — это локальный минимум из-за неудачного старта. Перезапустите ячейку с другим `rng`: результат изменится. Именно поэтому нужен k-means++.

## 2. Выбор k: локоть и силуэт (урок 6.2)

In [ ]:
ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    m = KMeans(k, n_init=10, random_state=0).fit(X)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(X, m.labels_))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ks, inertias, "o-")
ax[0].set_title("Инерция")
ax[1].plot(ks, sils, "o-", color="green")
ax[1].set_title("Средний силуэт")
plt.show()
print("Лучшее k по силуэту:", ks[int(np.argmax(sils))])

## 3. Сжатие цветов фотографии (урок 6.2)

Фотография из примеров scikit-learn: 427 × 640 пикселей, у каждого — три числа от 0 до 255.

**Задание 2.** Обучите `KMeans` с `k = 16` на каждом 20-м пикселе и замените цвет **каждого** пикселя цветом его центра.

In [ ]:
img = load_sample_image("china.jpg")
pixels = img.reshape(-1, 3).astype(float)
print("Разных цветов в оригинале:", len(np.unique(img.reshape(-1, 3), axis=0)))
compressed = ...  # ваш код

In [ ]:
assert compressed.shape == img.shape
assert len(np.unique(compressed.reshape(-1, 3), axis=0)) <= 16
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(img)
ax[0].set_title("Оригинал")
ax[1].imshow(compressed)
ax[1].set_title("16 цветов")
for a in ax:
    a.axis("off")
plt.show()
print("Верно! Бит на пиксель: 24 → 4")

## 4. Якоря YOLO (урок 6.2)

Условные рамки из разметки: ширина и высота в пикселях для картинки 416 × 416 — высокие узкие (люди), широкие низкие (машины), мелкие и крупные объекты.

**Задание 3.** Допишите IoU рамки со всеми якорями при общем центре.

In [ ]:
def anchor_iou(box, anchors):
    return ...  # ваш код


assert np.allclose(anchor_iou(np.array([2, 2]), np.array([[2, 2], [1, 1], [4, 1]])), [1, 0.25, 1 / 3])
print("Верно!")

In [ ]:
w1 = rng.lognormal(np.log(40), 0.35, 200); w2 = rng.lognormal(np.log(120), 0.35, 200)
w3 = rng.lognormal(np.log(18), 0.4, 100); w4 = rng.lognormal(np.log(260), 0.2, 100)
boxes = np.clip(np.c_[np.r_[w1, w2, w3, w4],
                      np.r_[w1 * rng.normal(2.4, 0.3, 200), w2 * rng.normal(0.6, 0.1, 200),
                            w3 * rng.normal(1.0, 0.2, 100), w4 * rng.normal(0.85, 0.15, 100)]], 4, 416)


def kmeans_iou(boxes, k, seed=0):
    anchors = boxes[np.random.default_rng(seed).choice(len(boxes), k, replace=False)]
    for _ in range(100):
        labels = np.array([np.argmax(anchor_iou(b, anchors)) for b in boxes])  # ближайший = наибольший IoU
        new = np.array([boxes[labels == j].mean(0) if np.any(labels == j) else anchors[j] for j in range(k)])
        if np.allclose(new, anchors):
            break
        anchors = new
    return anchors


def mean_best_iou(boxes, anchors):
    return np.mean([anchor_iou(b, anchors).max() for b in boxes])


anchors = kmeans_iou(boxes, 5)
euclid = KMeans(5, random_state=0).fit(boxes).cluster_centers_
print("Якоря (1 − IoU):", np.round(anchors[np.argsort(anchors.prod(1))]).astype(int).tolist())
print(f"Средний лучший IoU: 1 − IoU — {mean_best_iou(boxes, anchors):.3f}, евклидово — {mean_best_iou(boxes, euclid):.3f}")

## 5. DBSCAN (урок 6.3)

Полумесяцы и 20 случайных «одиноких звёзд». Сначала график расстояний до 5-го соседа: `eps` берут там, где кривая резко идёт вверх.

In [ ]:
Xm, _ = make_moons(300, noise=0.06, random_state=1)
Xn = np.r_[Xm, rng.uniform([-1.5, -1], [2.5, 1.5], (20, 2))]
dist, _ = NearestNeighbors(n_neighbors=5).fit(Xn).kneighbors(Xn)
plt.plot(np.sort(dist[:, -1]))
plt.ylabel("расстояние до 5-го соседа")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, eps in zip(axes, (0.05, 0.2, 0.5)):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xn)
    ax.scatter(*Xn.T, c=lab, cmap="tab10", s=8)
    ax.set_title(f"eps = {eps}: кластеров {lab.max() + 1}, шум {np.sum(lab == -1)}")
lab = HDBSCAN(min_cluster_size=15).fit_predict(Xn)
axes[3].scatter(*Xn.T, c=lab, cmap="tab10", s=8)
axes[3].set_title(f"HDBSCAN: кластеров {lab.max() + 1}, шум {np.sum(lab == -1)}")
plt.show()

## 6. PCA через SVD (урок 6.4)

**Задание 4.** Сделайте PCA своими руками через SVD: центрируйте `X`, разложите `np.linalg.svd(Xc, full_matrices=False)`, координаты в `k` компонентах — `Xc @ Vt[:k].T`, доли дисперсии — `s**2 / (s**2).sum()`.

In [ ]:
def pca_svd(X, k):
    return ..., ...  # ваш код: координаты (n, k) и доли дисперсии первых k компонент

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Z, ratio = pca_svd(Xd, 2)
sk = PCA(2).fit(Xd)
assert np.allclose(ratio, sk.explained_variance_ratio_)
assert np.allclose(np.abs(Z), np.abs(sk.transform(Xd)))  # знак компоненты может отличаться
print("Верно! Доли дисперсии:", ratio.round(3))
plt.scatter(*Z.T, c=yd, cmap="tab10", s=5)
plt.colorbar()
plt.title("Цифры в первых двух главных компонентах")
plt.show()

Сколько компонент нужно, чтобы сохранить 90 % дисперсии?

In [ ]:
cum = np.cumsum(PCA().fit(Xd).explained_variance_ratio_)
print("Компонент для 90 %:", np.searchsorted(cum, 0.9) + 1)

## 7. t-SNE и UMAP (урок 6.5)

Цвет — настоящая цифра; сами методы её не видят. Попробуйте другие `perplexity` (5, 100) и `random_state` — какие острова устойчивы?

In [ ]:
Zt = TSNE(2, perplexity=30, random_state=0).fit_transform(Xd)
plt.scatter(*Zt.T, c=yd, cmap="tab10", s=5)
plt.colorbar()
plt.title("t-SNE, perplexity = 30")
plt.show()

UMAP — отдельный пакет. Если ячейка ниже пишет, что его нет, выполните `%pip install umap-learn` и перезапустите её.

In [ ]:
try:
    import umap

    Zu = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Xd)
    plt.scatter(*Zu.T, c=yd, cmap="tab10", s=5)
    plt.colorbar()
    plt.title("UMAP")
    plt.show()
except ImportError:
    print("Пакета umap-learn нет: выполните %pip install umap-learn")